<a href="https://colab.research.google.com/github/pyhugeant/AIHC5045/blob/main/AIHC5045_module2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import the drive library from google.colab
from google.colab import drive

# start the mounting process
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display

In [7]:
# load the dataset
df = pd.read_csv("/content/drive/MyDrive/AIHC5045/06-02 Clinical validation study-level results.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
display(df.head())

Dataset shape: (3000, 12)

Column names:
['study_id', 'site', 'scanner_slices', 'slice_mm', 'series', 'finding', 'occlusion_site', 'ich_type', 'age_band', 'sex', 'stroke_flag', 'time_to_flag_s']


,study_id,site,scanner_slices,slice_mm,series,finding,occlusion_site,ich_type,age_band,sex,stroke_flag,time_to_flag_s
0,SV34-00001,B,128,1.000,NCCT,none,NaN,NaN,75-84,F,0,NaN
1,SV34-00002,C,128,1.000,NCCT,ICH,NaN,IPH,60-74,F,1,92.0
2,SV34-00003,D,256,0.625,CTA,none,NaN,NaN,45-59,M,0,NaN
3,SV34-00004,A,320,0.625,CTA,none,NaN,NaN,85+,M,0,NaN
4,SV34-00005,A,256,0.625,CTA,LVO,ICA,NaN,75-84,F,1,107.0


In [12]:
from math import sqrt

def wilson_ci(successes, total, z=1.959963984540054):
    """
    Return the 95% Wilson confidence interval for a proportion.
    successes: number of detected positive cases
    total: total reference-positive cases
    """
    if total == 0:
        return np.nan, np.nan

    p = successes / total
    denom = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denom
    half_width = (
        z * sqrt(p * (1 - p) / total + z**2 / (4 * total**2))
        / denom
    )

    return max(0, center - half_width), min(1, center + half_width)

In [13]:
def summarize_sensitivity(data, group_cols):
    """
    Calculate sensitivity and 95% Wilson CI by subgroup.

    Assumes:
      - reference-positive cases are defined by the selected finding
      - stroke_flag == 1 means the algorithm flagged the study
    """
    results = []

    grouped = data.groupby(group_cols, dropna=False)

    for group_key, group in grouped:
        n_positive = len(group)
        detected = int((group["stroke_flag"] == 1).sum())
        missed = int((group["stroke_flag"] == 0).sum())

        sensitivity = detected / n_positive if n_positive else np.nan
        ci_low, ci_high = wilson_ci(detected, n_positive)

        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        row = dict(zip(group_cols, group_key))
        row.update({
            "positive_cases_n": n_positive,
            "detected_n": detected,
            "missed_n": missed,
            "sensitivity": sensitivity,
            "sensitivity_pct": sensitivity * 100,
            "ci95_low_pct": ci_low * 100,
            "ci95_high_pct": ci_high * 100
        })

        results.append(row)

    return pd.DataFrame(results)

In [14]:
# ICH: NCCT studies with reference finding ICH
ich_positive = df[
    (df["series"] == "NCCT") &
    (df["finding"] == "ICH")
].copy()

# LVO: CTA studies with reference finding LVO
lvo_positive = df[
    (df["series"] == "CTA") &
    (df["finding"] == "LVO")
].copy()

ich_by_site = summarize_sensitivity(ich_positive, ["site"])
lvo_by_site = summarize_sensitivity(lvo_positive, ["site"])

print("ICH sensitivity by site")
display(ich_by_site.round(2))

print("LVO sensitivity by site")
display(lvo_by_site.round(2))

ICH sensitivity by site


,site,positive_cases_n,detected_n,missed_n,sensitivity,sensitivity_pct,ci95_low_pct,ci95_high_pct
0,A,188,171,17,0.91,90.96,86.00,94.28
1,B,138,131,7,0.95,94.93,89.90,97.52
2,C,118,107,11,0.91,90.68,84.08,94.71
3,D,136,131,5,0.96,96.32,91.68,98.42


LVO sensitivity by site


,site,positive_cases_n,detected_n,missed_n,sensitivity,sensitivity_pct,ci95_low_pct,ci95_high_pct
0,A,146,132,14,0.90,90.41,84.55,94.20
1,B,102,93,9,0.91,91.18,84.08,95.29
2,C,83,74,9,0.89,89.16,80.66,94.19
3,D,109,97,12,0.89,88.99,81.74,93.59


In [15]:
ich_by_type = summarize_sensitivity(
    ich_positive.dropna(subset=["ich_type"]),
    ["ich_type"]
)

display(
    ich_by_type.sort_values("sensitivity").round(2)
)

,ich_type,positive_cases_n,detected_n,missed_n,sensitivity,sensitivity_pct,ci95_low_pct,ci95_high_pct
4,SDH,150,136,14,0.91,90.67,84.94,94.36
3,SAH,100,91,9,0.91,91.00,83.77,95.19
2,IVH,40,37,3,0.92,92.50,80.14,97.42
0,EDH,40,38,2,0.95,95.00,83.50,98.62
1,IPH,250,238,12,0.95,95.20,91.80,97.23


In [16]:
lvo_by_site_type = summarize_sensitivity(
    lvo_positive.dropna(subset=["occlusion_site"]),
    ["occlusion_site"]
)

display(
    lvo_by_site_type.sort_values("sensitivity").round(2)
)

,occlusion_site,positive_cases_n,detected_n,missed_n,sensitivity,sensitivity_pct,ci95_low_pct,ci95_high_pct
4,vertebral,5,3,2,0.60,60.00,23.07,88.24
2,M2,24,15,9,0.62,62.50,42.71,78.84
3,basilar,11,7,4,0.64,63.64,35.38,84.83
0,ICA,118,108,10,0.92,91.53,85.10,95.33
1,M1,282,263,19,0.93,93.26,89.72,95.64


In [17]:
ich_by_config = summarize_sensitivity(
    ich_positive,
    ["scanner_slices", "slice_mm"]
)

lvo_by_config = summarize_sensitivity(
    lvo_positive,
    ["scanner_slices", "slice_mm"]
)

print("ICH sensitivity by scanner and slice thickness")
display(
    ich_by_config.sort_values("sensitivity").round(2)
)

print("LVO sensitivity by scanner and slice thickness")
display(
    lvo_by_config.sort_values("sensitivity").round(2)
)

ICH sensitivity by scanner and slice thickness


,scanner_slices,slice_mm,positive_cases_n,detected_n,missed_n,sensitivity,sensitivity_pct,ci95_low_pct,ci95_high_pct
0,64,1.25,65,59,6,0.91,90.77,81.29,95.70
4,320,0.62,104,95,9,0.91,91.35,84.37,95.38
2,256,0.62,174,162,12,0.93,93.10,88.33,96.01
1,128,1.00,191,179,12,0.94,93.72,89.34,96.37
3,256,1.00,46,45,1,0.98,97.83,88.66,99.62


LVO sensitivity by scanner and slice thickness


,scanner_slices,slice_mm,positive_cases_n,detected_n,missed_n,sensitivity,sensitivity_pct,ci95_low_pct,ci95_high_pct
3,256,1.00,39,32,7,0.82,82.05,67.33,91.02
0,64,1.25,45,39,6,0.87,86.67,73.82,93.74
4,320,0.62,75,67,8,0.89,89.33,80.34,94.50
1,128,1.00,140,128,12,0.91,91.43,85.62,95.03
2,256,0.62,141,130,11,0.92,92.20,86.57,95.59


In [18]:
def format_submission_table(summary, group_cols):
    out = summary.copy()

    out["Result"] = (
        out["detected_n"].astype(str)
        + "/"
        + out["positive_cases_n"].astype(str)
    )

    out["Rate"] = out["sensitivity_pct"].map(
        lambda x: f"{x:.1f}%" if pd.notna(x) else "NA"
    )

    out["95% CI"] = out.apply(
        lambda r: (
            f"{r['ci95_low_pct']:.1f}%–{r['ci95_high_pct']:.1f}%"
            if pd.notna(r["ci95_low_pct"])
            else "NA"
        ),
        axis=1
    )

    columns = group_cols + [
        "positive_cases_n",
        "Result",
        "Rate",
        "95% CI"
    ]

    return out[columns].rename(
        columns={"positive_cases_n": "Cases in group"}
    )

display(format_submission_table(ich_by_site, ["site"]))
display(format_submission_table(lvo_by_site, ["site"]))
display(format_submission_table(lvo_by_site_type, ["occlusion_site"]))

,site,Cases in group,Result,Rate,95% CI
0,A,188,171/188,91.0%,86.0%–94.3%
1,B,138,131/138,94.9%,89.9%–97.5%
2,C,118,107/118,90.7%,84.1%–94.7%
3,D,136,131/136,96.3%,91.7%–98.4%


,site,Cases in group,Result,Rate,95% CI
0,A,146,132/146,90.4%,84.5%–94.2%
1,B,102,93/102,91.2%,84.1%–95.3%
2,C,83,74/83,89.2%,80.7%–94.2%
3,D,109,97/109,89.0%,81.7%–93.6%


,occlusion_site,Cases in group,Result,Rate,95% CI
0,ICA,118,108/118,91.5%,85.1%–95.3%
1,M1,282,263/282,93.3%,89.7%–95.6%
2,M2,24,15/24,62.5%,42.7%–78.8%
3,basilar,11,7/11,63.6%,35.4%–84.8%
4,vertebral,5,3/5,60.0%,23.1%–88.2%


In [22]:
ich_by_site.to_csv("ICH_by_site.csv", index=False)
lvo_by_site.to_csv("LVO_by_site.csv", index=False)
ich_by_type.to_csv("ICH_by_type.csv", index=False)
lvo_by_site_type.to_csv("LVO_by_occlusion_site.csv", index=False)
ich_by_config.to_csv("ICH_by_configuration.csv", index=False)
lvo_by_config.to_csv("LVO_by_configuration.csv", index=False)

files.download("ICH_by_site.csv")
files.download("LVO_by_site.csv")
files.download("ICH_by_type.csv")
files.download("LVO_by_occlusion_site.csv")
files.download("ICH_by_configuration.csv")
files.download("LVO_by_configuration.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>